In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from google.cloud import bigquery
from google.oauth2 import service_account
import warnings
warnings.filterwarnings("ignore")

print("⏳ KHỞI TẠO KIẾN TRÚC DỮ LIỆU TRÁI PHIẾU...")

# ==============================================================
# BẢNG 1 (DIM): SỔ CÁI DANH MỤC TRÁI PHIẾU (Cấu hình cố định)
# ==============================================================
bond_data = {
    "Bond_ID": ["MSN2026", "VIC2027", "HPG2028", "TCB2025", "VHM2029"],
    "Issuer": ["Masan", "Vingroup", "Hoa Phat", "Techcombank", "Vinhomes"],
    "Face_Value": [1000000000, 1000000000, 1000000000, 1000000000, 1000000000],
    "Coupon_Rate": [0.085, 0.092, 0.078, 0.065, 0.095], 
    "Issue_Date": ["2023-01-01", "2024-05-15", "2023-10-10", "2022-12-01", "2024-01-20"],
    "Maturity_Date": ["2026-12-31", "2027-05-15", "2028-10-10", "2025-12-01", "2029-01-20"],
    "Payment_Freq": [1, 2, 1, 1, 2], 
    "Z_Spread": [0.025, 0.030, 0.015, 0.012, 0.035] # Chìa khóa sinh tử để định giá
}
df_dim_bonds = pd.DataFrame(bond_data)
# Ép kiểu Datetime thành String cho BigQuery
df_dim_bonds['Issue_Date'] = pd.to_datetime(df_dim_bonds['Issue_Date']).astype(str)
df_dim_bonds['Maturity_Date'] = pd.to_datetime(df_dim_bonds['Maturity_Date']).astype(str)

# ==============================================================
# BẢNG 2 (FACT): ĐƯỜNG CONG LỢI SUẤT HÀNG NGÀY (Cập nhật liên tục)
# ==============================================================
today = datetime.now().strftime('%Y-%m-%d')
dates = pd.date_range(start="2024-01-01", end=today, freq="B")

# Thuật toán Random Walk giả lập biến động lợi suất chuẩn mực (Đi lên)
np.random.seed(42)
yield_1Y = 0.045 + np.cumsum(np.random.normal(0, 0.0002, len(dates)))
yield_2Y = yield_1Y + 0.005 + np.random.normal(0, 0.0001, len(dates))
yield_3Y = yield_2Y + 0.004 + np.random.normal(0, 0.0001, len(dates))
yield_5Y = yield_3Y + 0.006 + np.random.normal(0, 0.0001, len(dates))
yield_10Y = yield_5Y + 0.010 + np.random.normal(0, 0.0001, len(dates))

df_fact_yield = pd.DataFrame({
    "Date": dates.astype(str),
    "Yield_1Y": yield_1Y, "Yield_2Y": yield_2Y, "Yield_3Y": yield_3Y,
    "Yield_5Y": yield_5Y, "Yield_10Y": yield_10Y
})

# ==============================================================
# Đẩy LÊN BIGQUERY
# ==============================================================
project_id = "jda-k1"
dataset_id = "cuongnm_data_pipeline"
key_path = r"C:\Users\DELL\Desktop\Project\Project Market risk\Key - path 2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

print("⏳ Bắn Sổ cái Trái phiếu (DIM) lên BigQuery...")
client.load_table_from_dataframe(df_dim_bonds, f"{project_id}.{dataset_id}.DIM_Bond_Portfolio", job_config=job_config).result()

print("⏳ Bắn Đường cong Lợi suất (FACT) lên BigQuery...")
client.load_table_from_dataframe(df_fact_yield, f"{project_id}.{dataset_id}.FACT_Yield_Curve", job_config=job_config).result()

print("✅ XUẤT SẮC! Dữ liệu Trái phiếu đã yên vị trên Đám mây!")

⏳ KHỞI TẠO KIẾN TRÚC DỮ LIỆU TRÁI PHIẾU...
⏳ Bắn Sổ cái Trái phiếu (DIM) lên BigQuery...
⏳ Bắn Đường cong Lợi suất (FACT) lên BigQuery...
✅ XUẤT SẮC! Dữ liệu Trái phiếu đã yên vị trên Đám mây!
